# LC 300 — Longest Increasing Subsequence
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Dynamic Programming
**Pattern:** 1D DP — Look Back / Binary Search on Tails

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> dp[i] = length of the
longest increasing subsequence that ends at index i.
For every earlier j where nums[j] < nums[i], you can
extend dp[j] by one. The O(n log n) upgrade keeps a
"tails" array and binary-searches into it.
</div>

## Official Problem Statement

Given an integer array `nums`, return the length of
the longest strictly increasing subsequence.

**Example 1:**
```
Input:  nums = [10,9,2,5,3,7,101,18]
Output: 4
Explanation: [2,3,7,101] or [2,5,7,101]
```
**Example 2:**
```
Input:  nums = [0,1,0,3,2,3]
Output: 4
```
**Example 3:**
```
Input:  nums = [7,7,7,7,7,7,7]
Output: 1
```

**Constraints:**
- `1 <= nums.length <= 2500`
- `-10^4 <= nums[i] <= 10^4`

## What This Is Actually Asking

Pick a subset of numbers from the list — order must
be preserved (you can't rearrange them) — such that
each number in the subset is strictly bigger than
the one before it.
You can skip any numbers; you just can't reorder.
Find the length of the longest such subset you can
build.

## Walk Through an Example by Hand

```
nums = [10, 9, 2, 5, 3, 7, 101, 18]
index:   0  1  2  3  4  5    6   7

dp[i] = length of LIS ending at index i

dp[0] = 1  (just [10])
dp[1] = 1  (just [9], 10 is not < 9)
dp[2] = 1  (just [2])
dp[3] = 2  (2 < 5 -> extend dp[2]: [2,5])
dp[4] = 2  (2 < 3 -> extend dp[2]: [2,3])
dp[5] = 3  (2<7 -> dp[2]+1=2, 5<7 -> dp[3]+1=3,
             3<7 -> dp[4]+1=3 -> dp[5]=3: [2,5,7])
dp[6] = 4  (all smaller -> best is dp[5]+1: [2,5,7,101])
dp[7] = 4  (2<18->2, 5<18->3, 3<18->3, 7<18->4)

Answer: max(dp) = 4
```

## The Picture

```
--- O(n^2) DP: "look back" ---

nums: [10,  9,  2,  5,  3,  7, 101,  18]
dp:  [  1,  1,  1,  2,  2,  3,   4,   4]
                 ^   ^       ^    ^answer
      for each i, scan all j<i where nums[j]<nums[i]
      dp[i] = max(dp[j]+1) across those j

--- O(n log n) TAILS: "patience sort" ---

tails[] = smallest known tail for each LIS length

num=10: tails=[]   -> append 10 -> [10]
num= 9: bisect([10],9)=0 -> replace -> [9]
num= 2: replace -> [2]
num= 5: append  -> [2,5]       (LIS length 2 exists)
num= 3: replace tails[1] -> [2,3]
num= 7: append  -> [2,3,7]     (LIS length 3 exists)
num=101: append -> [2,3,7,101] (LIS length 4!)
num=18: replace tails[3] -> [2,3,7,18]

Answer: len(tails) = 4

tails is NOT the actual LIS — just its length.
Replacing keeps each slot's tail as small as
possible, making future extensions more likely.
```

## When To Use This Pattern

- When you see **longest strictly increasing
  subsequence**, think **dp[i] = max(dp[j]+1)**
  or the **tails binary search** upgrade
- When O(n²) is too slow (n > 2500), think
  **bisect on tails[] for O(n log n)**
- When asked for the length only (not the actual
  subsequence), think **tails is enough**
- When extending to non-decreasing (not strictly),
  think **use bisect_right instead of bisect_left**

## The Approach

**O(n²) DP:** Initialize every dp[i] to 1. For each
index i, scan all earlier indices j; if nums[j] is
less than nums[i], dp[i] could extend that sequence.
Return the maximum value in dp.

**O(n log n) Tails:** Maintain a tails list where
tails[k] is the smallest tail of all LIS of length
k+1 seen so far. For each number, binary-search for
its position in tails — if it extends beyond the
end, append; otherwise replace. The length of tails
is the answer.

In [ ]:
from typing import List   # type hints for the solution
import bisect             # binary search on sorted list

In [ ]:
def test_harness(func):
    tests = [
        # (nums, expected)
        ([10,9,2,5,3,7,101,18], 4),  # [2,3,7,101]
        ([0,1,0,3,2,3],         4),  # [0,1,2,3]
        ([7,7,7,7,7,7,7],       1),  # all equal
        ([1],                   1),  # single element
        ([1,2,3,4,5],           5),  # already sorted
        ([5,4,3,2,1],           1),  # descending
        ([3,5,6,2,5,4,19,5,6,7,12], 6),  # mixed
        ([1,3,6,7,9,4,10,5,6],  6),
    ]

    passed = 0
    for i, (nums, expected) in enumerate(tests):
        result = func(nums[:])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"nums={nums} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def lengthOfLIS(nums: List[int]) -> int:
    """
    Return length of longest strictly increasing
    subsequence in nums.

    Maintain tails[]: tails[k] = smallest tail of
    any LIS of length k+1 seen so far. For each num,
    binary search (bisect_left) for its position in
    tails. Append if it extends the longest known
    sequence; replace otherwise. len(tails) = answer.

    Time:  O(n log n) — one bisect per element
    Space: O(n) — tails list, at most n elements
    """
    pass


# Quick debug — run this cell while building
print(lengthOfLIS([10,9,2,5,3,7,101,18]))  # 4
print(lengthOfLIS([0,1,0,3,2,3]))          # 4
print(lengthOfLIS([7,7,7,7]))              # 1
print(lengthOfLIS([1,2,3,4,5]))            # 5

In [ ]:
# Uncomment and run when solution is ready
# test_harness(lengthOfLIS)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — check all subsequences | O(2^n) | O(n) |
| DP look-back | O(n²) | O(n) |
| Binary search on tails | O(n log n) | O(n) |

The tails binary search is the interview upgrade —
same O(n) space but log-linear time, critical when
n can reach 2500 or more.

## Real World Connection

At Citi, detecting sustained CPU growth trends in
the telemetry pipeline is an LIS problem: given a
series of daily average CPU readings, the longest
strictly increasing subsequence identifies the
longest period of genuine, unbroken capacity
growth — a signal that auto-scaling or server
provisioning should be triggered.
The O(n log n) tails approach runs efficiently on
the full 90-day rolling window (thousands of data
points per server) without a performance bottleneck.
The same pattern applies to version-compatibility
chains: find the longest sequence of library
versions where each is strictly newer than the
previous — useful for dependency resolution in the
CI/CD pipeline.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra